# LangGraph + TypeSafe

**Difficulty:** Intermediate

LangGraph orchestrates a workflow as a graph of nodes and edges; TypeSafe makes narrow, structured decisions. Neither one needs the other, but they compose well at one specific seam: LangGraph's *conditional edges* have to turn "what does this state mean" into the name of the next node to run, and a TypeSafe `Choice` — a closed-set answer with a `confidence` score — is a better fit for that than parsing free-text LLM output by hand.

This cookbook is original to this repo, not a `docs.typesafe.ai` cookbook — there isn't one. It's grounded in LangGraph's own docs (see `research/langgraph.md`) plus this project's own confidence-gated routing and intent-routing lessons.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.langchain.com/oss/javascript/langgraph — overview + graph-api (click to expand)</summary>

# LangGraph

> LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents. LangGraph gives you fine-grained control to mix deterministic, hand-coded steps with LLM-driven agentic steps in the same graph.

Install: `npm install @langchain/langgraph @langchain/core`.

## Graphs

At its core, LangGraph models agent workflows as graphs, defined with three components:

1. **State**: a shared data structure representing the current snapshot of the application, typically defined with a shared state schema.
2. **Nodes**: functions that encode the logic of your agents. They receive the current state as input, perform some computation, and return an updated state.
3. **Edges**: functions that determine which node to execute next based on the current state. They can be conditional branches or fixed transitions.

*Nodes do the work, edges tell what to do next.* `Nodes` and `Edges` are nothing more than functions — they can contain an LLM call or just ordinary code.

### StateGraph and compiling

`StateGraph` is the main graph class, parameterized by a `StateSchema`. You define state, add nodes and edges, then compile: "You **MUST** compile your graph before you can use it."

```typescript
const graph = new StateGraph(State)
  .addNode("nodeA", nodeA)
  .addEdge(START, "nodeA")
  .addEdge("nodeA", END)
  .compile();
```

### State and reducers

Each field in a `StateSchema` has a reducer function specifying how a node's update is applied to it. By default, "the default reducer ignores the left argument and replaces the state value with the right argument" — an update simply overwrites. A custom reducer can accumulate instead (e.g. concatenating arrays), for cases where parallel nodes need to merge their updates rather than one clobbering another.

### Edges

- **Normal edges** (`addEdge("nodeA", "nodeB")`): always go from A to B.
- **Conditional edges** (`addConditionalEdges("nodeA", routingFunction)`): call a function to determine which node(s) to go to next. By default, the routing function's return value is used as the name of the next node; you can also supply a mapping from the function's return value to a node name.
- **Entry point** (`addEdge(START, "nodeA")`) and **conditional entry point** (`addConditionalEdges(START, routingFunction)`): which node (or which node, decided dynamically) runs first.

> For each node, choose one routing mechanism: use normal edges for static routing, or use conditional edges for dynamic routing. Do not mix normal edges and dynamic routing from the same node, because both paths can execute and make graph behavior harder to reason about.

The docs describe conditional edges mechanically — a function of `state` that returns a node name — but don't prescribe how that function should *decide*. That's the open seam this cookbook fills.

> The full guides also cover `Send` (fan-out to a variable number of node invocations), `Command` (bundling a state update with routing in one return value), persistence via checkpointers, and streaming — trimmed here to the vocabulary this cookbook actually uses. Read them at [docs.langchain.com/oss/javascript/langgraph/overview](https://docs.langchain.com/oss/javascript/langgraph/overview) and [.../graph-api](https://docs.langchain.com/oss/javascript/langgraph/graph-api) any time.

</details>

## The core idea

A conditional edge's routing function has to return the name of the next node — a value from a fixed, closed set. That's exactly the shape a TypeSafe `Choice` question already returns: pick one option from a defined set, plus a `confidence` for how sure the model was. Instead of asking an LLM a free-text question and parsing its reply to decide which branch to take, the routing function can read a `Choice` answer's `choice` field directly.

`confidence` earns its keep here too. This repo's confidence-gated routing (`lessons/patterns/0007-confidence-gated-routing.ipynb`) said a model's answer should decide not just *what* to do, but *whether* to act automatically at all. Applied to a graph: below some confidence floor, the routing function sends the state to a `humanReview` node regardless of which category won — the LangGraph equivalent of intent routing's escalation path (`lessons/patterns/0009-intent-routing.ipynb`), expressed as one more branch of a conditional edge instead of a separate `if` in application code.

Nothing about this needs LangGraph's more advanced machinery — no custom reducers, no persistence, no `Send`/`Command`. One `StateGraph`, one node that calls TypeSafe, one conditional edge whose routing function reads that node's output.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";
import { StateGraph, StateSchema, START, END } from "npm:@langchain/langgraph";
import type { GraphNode } from "npm:@langchain/langgraph";
import * as z from "npm:zod";

const CONFIDENCE_FLOOR = 0.6;

const State = new StateSchema({
  ticket: z.string(),
  category: z.string().optional(),
  confidence: z.number().optional(),
  resolution: z.string().optional(),
});

// The seam: one TypeSafe Choice call, its answer written straight into graph state.
const classify: GraphNode<typeof State> = async (state) => {
  const response = await client.systemOne({
    state: state.ticket,
    questions: {
      category: choice("Determine the broad category of this support ticket", {
        billing: "Charges, invoices, refunds, subscriptions",
        account: "Login, permissions, profile, security",
        feature_request: "The user is requesting new functionality",
      }),
    },
  });
  const { category } = response.answers;
  return { category: category.choice, confidence: category.confidence };
};

// The routing function: reads the Choice answer's confidence before trusting its category.
function route(state: typeof State.State): string {
  if ((state.confidence ?? 0) < CONFIDENCE_FLOOR) return "humanReview";
  return state.category ?? "humanReview";
}

const handleBilling: GraphNode<typeof State> = () => ({ resolution: "routed to billing team" });
const handleAccount: GraphNode<typeof State> = () => ({ resolution: "routed to account team" });
const handleFeature: GraphNode<typeof State> = () => ({ resolution: "logged as feature request" });
const humanReview: GraphNode<typeof State> = () => ({ resolution: "flagged for human review" });

const graph = new StateGraph(State)
  .addNode("classify", classify)
  .addNode("billing", handleBilling)
  .addNode("account", handleAccount)
  .addNode("feature_request", handleFeature)
  .addNode("humanReview", humanReview)
  .addEdge(START, "classify")
  .addConditionalEdges("classify", route, {
    billing: "billing",
    account: "account",
    feature_request: "feature_request",
    humanReview: "humanReview",
  })
  .addEdge("billing", END)
  .addEdge("account", END)
  .addEdge("feature_request", END)
  .addEdge("humanReview", END)
  .compile();

const tickets: Record<string, string> = {
  clearAccountIssue: "I can't log into my account since the password reset yesterday.",
  genuinelyAmbiguous: "Hey, something's off with my order and I'm not sure who to ask.",
};

for (const [name, ticket] of Object.entries(tickets)) {
  const result = await graph.invoke({ ticket });
  console.log(name, result);
}

## What just happened

Two tickets through the same graph. `clearAccountIssue` names a specific, ordinary account action (a password reset) — expect `classify` to land on `account` at high confidence, and the conditional edge to route straight to the `account` leaf node. `genuinelyAmbiguous` deliberately withholds any category signal ("something's off," no mention of a charge, a login, or a feature) — expect `confidence` to fall below `CONFIDENCE_FLOOR` regardless of which category narrowly wins, and the routing function to send it to `humanReview` instead of guessing.

The graph structure never changes between the two runs — the same five nodes and one conditional edge handle both. Only the state flowing through it differs. That's the LangGraph half of the point as much as the TypeSafe half: the routing *decision* is data (a `Choice` answer written into state), not a hardcoded branch, so the graph doesn't need a special case for "ambiguous ticket" — it already has one, because `confidence` is just another field on state like any other.

**Try it yourself:** predict both outcomes — category and route for each ticket — before running. Then add a fourth node and a fourth category (e.g. `spam`) and confirm you only need to touch the `classify` node's `Choice` criteria and the conditional edge's mapping — nothing else in the graph's shape changes.

## Related

- `lessons/patterns/0007-confidence-gated-routing.ipynb` — the confidence-floor idea this cookbook's `CONFIDENCE_FLOOR` applies inside a LangGraph conditional edge instead of a plain `if`
- `lessons/patterns/0009-intent-routing.ipynb` — the same classify-then-dispatch shape, without a graph framework wrapped around it
- `lessons/patterns/0006-speculative-fan-out.ipynb` — the ticket-triage example this cookbook's `classify` node is deliberately close to
- `research/langgraph.md` — the primary-source research this cookbook was built from, including what the LangGraph docs do and don't say about routing-function design

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/langgraph.md` and `research/typesafe-ai.md`.